<a href="https://colab.research.google.com/github/salmaelhanchi/NeuroMatch_2026_Behaviour/blob/main/data_preparation_and_hb_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data preparation & hierarchical model notebook

Continues directly from the data-loading cell Salma set up (`raw = pd.read_csv(...)`).
Each section below follows the same pattern: **what we tried** -> **what we found** -> **what's decided** (or still open).


In [1]:
import pandas as pd
from pathlib import Path

REPO_URL = "https://github.com/salmaelhanchi/NeuroMatch_2026_Behaviour.git"
REPO_DIR = Path("/content/NeuroMatch_2026_Behaviour")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

matches = list(REPO_DIR.rglob("data01_direction4priors.csv"))

if not matches:
    raise FileNotFoundError("data01_direction4priors.csv was not found inside the cloned repo.")

CSV_PATH = matches[0]
raw = pd.read_csv(CSV_PATH)

print(f"Loaded from: {CSV_PATH}")
print(f"raw shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head()


Cloning into '/content/NeuroMatch_2026_Behaviour'...
remote: Enumerating objects: 97, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 97 (delta 44), reused 51 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (97/97), 4.58 MiB | 3.70 MiB/s, done.
Resolving deltas: 100% (44/44), done.
Loaded from: /content/NeuroMatch_2026_Behaviour/data01_direction4priors.csv
raw shape: 83,213 rows x 16 columns


,trial_index,trial_time,response_arrow_start_angle,motion_direction,motion_coherence,estimate_x,estimate_y,reaction_time,raw_response_time,prior_std,prior_mean,subject_id,experiment_name,experiment_id,session_id,run_id
0,1,0.000000,NaN,225,0.12,-1.749685,-1.785666,NaN,NaN,10,225,1,data01_direction4priors,11,1,1
1,2,2.730730,NaN,225,0.12,-1.819693,-1.714269,NaN,NaN,10,225,1,data01_direction4priors,11,1,1
2,3,4.913950,NaN,235,0.06,-1.562674,-1.951422,NaN,NaN,10,225,1,data01_direction4priors,11,1,1
3,4,6.997296,NaN,225,0.06,-1.601388,-1.919781,NaN,NaN,10,225,1,data01_direction4priors,11,1,1
4,5,9.097130,NaN,215,0.24,-1.639461,-1.887371,NaN,NaN,10,225,1,data01_direction4priors,11,1,1


## 1. Building the trial-level dataset

`estimate_x`/`estimate_y` encode the subject's response as a coordinate pair, not a plain angle
(same idea as `motion_direction`, just represented differently on screen). `np.arctan2` recovers
the angle correctly because it looks at both signs at once -- plain `arctan(y/x)` can't tell
opposite directions apart, since they give the same ratio.

Circular error can't be plain subtraction either, for the same reason `error_deg` needed the
`% 360` trick: 355 degrees and 5 degrees are 10 degrees apart on the circle, not 350.


In [2]:
import numpy as np

data = raw.copy()

# --- convert response vector to an angle --------------------------------
data['estimate_deg'] = np.degrees(np.arctan2(data.estimate_y, data.estimate_x)) % 360
data.loc[data.estimate_deg == 0, 'estimate_deg'] = 360

n_before = len(data)
data = data.dropna(subset=['estimate_deg', 'motion_direction']).copy()
print(f"dropped {n_before - len(data)} rows with missing estimate")

# --- circular estimation error -------------------------------------------
# shift by 180 before the modulo, then shift back -- handles both directions
# of wraparound without an if-statement.
data['error_deg'] = ((data.estimate_deg - data.motion_direction + 180) % 360) - 180

# --- block id --------------------------------------------------------------
# run_id ALONE is not a unique block: it resets to 1 for every subject, so
# grouping by run_id alone lumps "subject 1's first block" together with
# "subject 2's first block". (subject_id, run_id) together is the real,
# unique block identifier -- combined here into one string column.
data['block_id'] = data['subject_id'].astype(str) + '_' + data['run_id'].astype(str)

trial_data = data.rename(columns={
    'subject_id': 'subject',
    'trial_index': 'trial',
    'prior_std': 'prior_width',
    'motion_coherence': 'coherence',
    'motion_direction': 'true_direction',
})[['subject', 'block_id', 'trial', 'prior_width', 'coherence',
    'true_direction', 'estimate_deg', 'error_deg',
    'session_id', 'experiment_id']]

print(f"trial_data shape: {trial_data.shape}")
trial_data.head()


dropped 3 rows with missing estimate
trial_data shape: (83210, 10)


,subject,block_id,trial,prior_width,coherence,true_direction,estimate_deg,error_deg,session_id,experiment_id
0,1,1_1,1,10,0.12,225,225.583113,0.583113,1,11
1,1,1_1,2,10,0.12,225,223.291282,-1.708718,1,11
2,1,1_1,3,10,0.06,235,231.312691,-3.687309,1,11
3,1,1_1,4,10,0.06,225,230.166776,5.166776,1,11
4,1,1_1,5,10,0.24,215,229.020860,14.020860,1,11


**What we tried:** grouping by `run_id` alone to check block sizes -- gave nonsense block
sizes in the thousands, because `run_id` resets per subject.

**What we found:** `(subject_id, run_id)` blocks range 204-226 trials (paper reports 202-226),
and each block has exactly one `prior_std` value. Confirmed below.

**Decided:** block = `(subject_id, run_id)`, stored as a single `block_id` string column.


In [3]:
block_sizes = data.groupby(['subject_id', 'run_id']).size()
print('block size range:', block_sizes.min(), '-', block_sizes.max())

prior_check = data.groupby(['subject_id', 'run_id'])['prior_std'].nunique()
print('blocks with more than one prior_std value (should be 0):', (prior_check > 1).sum())


block size range: 145 - 428
blocks with more than one prior_std value (should be 0): 3


## 2. What `experiment_id` actually is

**What we tried:** checked whether `experiment_id == 12` was a separate 5-subject follow-up
study, as originally theorized.

**What we found:** it isn't. `experiment_id` 12 appears for 10 of the 12 subjects (not 5), and
summing sessions across `experiment_id` 11 + 12 for each subject matches the paper's reported
session counts exactly (8, 8, 9, 5, 6, 7, 6, 6, 8, 6, 6, 6 for subjects 1-12). It looks like a
database/registration split of one single study, not two different studies.

**Decided:** treat `experiment_id` 11 and 12 as one combined dataset per subject. The
five-subject-follow-up theory is resolved as incorrect.


In [4]:
session_check = trial_data.groupby('subject')['session_id'].nunique()
print(session_check)
print('paper reports: 8, 8, 9, 5, 6, 7, 6, 6, 8, 6, 6, 6')


subject
1     8
2     8
3     9
4     5
5     6
6     7
7     6
8     6
9     8
10    6
11    6
12    6
Name: session_id, dtype: int64
paper reports: 8, 8, 9, 5, 6, 7, 6, 6, 8, 6, 6, 6


## 3. Data-quality caveat: `trial_index` and `trial_time` are block-relative, not global

**What we tried:** measure the real time gap between the last trial of one block and the first
trial of the next, to check whether blocks run back-to-back within a session (this matters for
the carryover hypothesis -- if there's no break, "bleeding" between blocks is physically
plausible).

**What we found:** both `trial_index` and `trial_time` reset to 1 / 0.0 at the start of every
block. Sorting a subject's rows by raw `trial_index` does **not** give true chronological order,
it groups "everyone's trial #1 of any block" together. This also means the raw file has **no way
to measure the real wall-clock gap between blocks** -- `trial_time` resets right at the boundary,
erasing that information. (Confirmed directly below for subject 2.)

**Decided (for now):** rely on the paper's Methods text (no rest period is described between
blocks within a session) as the best available evidence that blocks run consecutively.

**Still open:** confirm with Salma/Varsha whether a source with real inter-block timestamps
exists anywhere outside this CSV. Any feature that depends on trial order (previous block,
previous trial, RT trends) must sort by `(run_id, trial_index)` within a subject, never raw
`trial_index` alone -- and subjects with data split across two `experiment_id`s (1, 3, 5, 7, 8)
need extra care, since `trial_index` isn't unique across that split either.


In [5]:
demo = data[data.subject_id == 2].sort_values(['run_id', 'trial_index'])
print(demo[['run_id', 'trial_index', 'trial_time']].iloc[208:214].to_string(index=False))


 run_id  trial_index  trial_time
      1          209  545.224488
      1          210  547.740149
      1          211  550.116114
      1          212  552.532456
      1          213  554.937880
      2            1    0.000000


## 4. Hierarchical model: reliability-controlled switching

Varsha's settled hypothesis (13 Jul meeting): *"The observer first estimates how reliable the
prior itself is, and this estimate (hyper-prior) controls switching."*

**What we tried first (rejected):** a continuous-width hyperprior that multiplies prior x
likelihood together (the model in `hb_verified_model_implementation.ipynb`, using
`prior_kappa_t`). This is unimodal by construction -- a product of two von Mises distributions
is always a single von Mises -- so it cannot reproduce the paper's central bimodality finding.
The team explicitly moved away from this construction.

**What we're building instead:** a discrete mixture. Each trial, the observer reports from the
prior component with probability `prior_reliance`, or from the likelihood component with
probability `1 - prior_reliance`. `prior_reliance` is a hidden state, learned trial-by-trial via
a delta rule (deliberately not named `prior_confidence`, since that name is already used by the
rejected continuous-width version -- reusing it would blur two different mechanisms together).


In [6]:
# borrowed from switching_bayesian_observer_starter.py -- needed by trial_percept_distribution
from scipy.special import i0e

DEG = np.arange(1, 361)

def vm_pdf(support_deg, mu_deg, k, norm=True):
    mu = np.atleast_1d(np.asarray(mu_deg, float))
    x  = np.deg2rad(np.asarray(support_deg, float))[:, None]
    u  = np.deg2rad(mu)[None, :]
    k  = float(k)
    if np.isinf(k) or k > 1e300:
        out = np.zeros((len(support_deg), len(mu)))
        for j, mm in enumerate(mu):
            out[np.argmin(np.abs(np.asarray(support_deg) - mm)), j] = 1.0
    else:
        out = np.exp(k * np.cos(x - u) - k) / (2 * np.pi * i0e(k))
    if norm:
        out = out / out.sum(0, keepdims=True)
    return out


In [7]:
def wrap_signed_deg(diff_deg):
    # same formula derived by hand for error_deg
    return ((diff_deg + 180) % 360) - 180


def circular_mean_deg(angles_deg):
    """
    Circular mean of several angles. Can't average degrees directly, same
    wraparound issue as error_deg, so convert to (x, y) unit vectors first,
    average those, then convert back with arctan2.
    """
    angles_rad = np.deg2rad(np.asarray(angles_deg))
    mean_x = np.mean(np.cos(angles_rad))
    mean_y = np.mean(np.sin(angles_rad))
    return float(np.degrees(np.arctan2(mean_y, mean_x)) % 360)


def prior_agreement(measurement_deg, prior_mean_deg, k_prior):
    """
    How consistent was this trial's sensory measurement with what the prior
    expected, on a 0-to-1 scale? 1.0 = measurement landed exactly on the
    prior mean. Decays toward 0 as the measurement moves away.
    """
    delta_rad = np.deg2rad(wrap_signed_deg(measurement_deg - prior_mean_deg))
    return float(np.exp(k_prior * (np.cos(delta_rad) - 1.0)))


def update_prior_reliance(prior_reliance, measurement_deg, prior_mean_deg, k_prior, alpha):
    """
    Delta-rule update for prior_reliance: how much the observer currently
    relies on the prior, learned trial by trial, in [0, 1].

        prior_reliance_next = prior_reliance + alpha * (agreement_t - prior_reliance)
    """
    agreement_t = prior_agreement(measurement_deg, prior_mean_deg, k_prior)
    reliance_next = prior_reliance + alpha * (agreement_t - prior_reliance)
    return float(np.clip(reliance_next, 1e-4, 1.0 - 1e-4)), agreement_t


def trial_percept_distribution(prior_mean_deg, measurement_deg, k_prior, k_llh, prior_reliance):
    """
    P(percept | trial) as a genuine MIXTURE (added, never multiplied), so two
    peaks survive whenever prior_reliance sits away from 0 or 1:

        P(percept) = prior_reliance       * VonMises(percept; prior_mean,  k_prior)
                   + (1 - prior_reliance) * VonMises(percept; measurement, k_llh)
    """
    prior_component = vm_pdf(DEG, prior_mean_deg, k_prior)[:, 0]
    llh_component   = vm_pdf(DEG, measurement_deg, k_llh)[:, 0]
    mixture = prior_reliance * prior_component + (1 - prior_reliance) * llh_component
    return mixture / mixture.sum()


### Sanity check: does `prior_reliance` move the right direction?

Toy sequence: three trials agreeing with the prior, three clashing, three agreeing again.


In [8]:
prior_mean, k_prior, alpha = 225, 8, 0.2
prior_reliance = 0.5

measurements = [225, 220, 230,   45, 50, 40,   228, 222, 226]
for m in measurements:
    prior_reliance, agreement_t = update_prior_reliance(prior_reliance, m, prior_mean, k_prior, alpha)
    print(f"measurement={m:>3}  agreement={agreement_t:.3f}  prior_reliance={prior_reliance:.3f}")


measurement=225  agreement=1.000  prior_reliance=0.600
measurement=220  agreement=0.970  prior_reliance=0.674
measurement=230  agreement=0.970  prior_reliance=0.733
measurement= 45  agreement=0.000  prior_reliance=0.587
measurement= 50  agreement=0.000  prior_reliance=0.469
measurement= 40  agreement=0.000  prior_reliance=0.375
measurement=228  agreement=0.989  prior_reliance=0.498
measurement=222  agreement=0.989  prior_reliance=0.596
measurement=226  agreement=0.999  prior_reliance=0.677


**Result:** reliance climbed 0.5 -> 0.73 across three agreeing trials, dropped to 0.375
across three clashing trials, climbed back to 0.68 once agreement resumed. Direction of movement
is correct.

### Real block-transition test

Used the paper's own concentration values (STAR Methods): k=33.3 for a 10-degree prior, k=0.7
for an 80-degree prior. Simulated 20 trials of a narrow block followed by 20 trials of a wide
block, drawing directions from `np.random.default_rng(7).vonmises(...)` for realism.

**Found:** with `alpha=0.2`, `prior_reliance` swung noisily (0.44-0.84) even during the
genuinely-predictable narrow block, single noisy trials could yank reliance around, because the
yardstick for "surprising" comes from the same sharp `k_prior` as the block's true concentration.


In [9]:
rng = np.random.default_rng(7)
k_prior_10, k_prior_80 = 33.3, 0.7
prior_mean_val = 225
mu_rad = np.deg2rad(prior_mean_val)

block_A = (np.rad2deg(rng.vonmises(mu_rad, k_prior_10, 20)) + 360) % 360
block_B = (np.rad2deg(rng.vonmises(mu_rad, k_prior_80, 20)) + 360) % 360

reliance = 0.5
trajectory = []
for m in block_A:
    reliance, _ = update_prior_reliance(reliance, m, prior_mean_val, k_prior_10, alpha=0.2)
    trajectory.append(reliance)
for m in block_B:
    reliance, _ = update_prior_reliance(reliance, m, prior_mean_val, k_prior_80, alpha=0.2)
    trajectory.append(reliance)

print('peak in 10-degree block:', round(max(trajectory[:20]), 3))
print('worst dip in 10-degree block:', round(min(trajectory[:20]), 3))
print('range in 80-degree block:', round(max(trajectory[20:]) - min(trajectory[20:]), 3))


peak in 10-degree block: 0.838
worst dip in 10-degree block: 0.436
range in 80-degree block: 0.206


**Tried lowering alpha -- a real tradeoff, not a fix.** Lower alpha calmed the noise (range
narrowed from a 0.4-point swing down to about 0.05 at `alpha=0.05`) but also weakened how high
`prior_reliance` could climb within a typical block, a genuine speed-vs-stability tradeoff, not
resolved by tuning alpha alone.

**Tried smoothing agreement over a short recent-trial window instead -- this worked better.**
Judging agreement against the circular mean of the last 5 measurements (not the newest one
alone) let `prior_reliance` climb fast (still `alpha=0.2`) *and* stay stable: peak 0.95 in the
narrow block (vs. 0.84 before), worst dip 0.58 (vs. 0.44 before).

**Open design question:** in that test the 5-trial window wasn't reset at the block boundary, so
a few trials of memory naturally bled across into the new block -- an unplanned but plausible
proxy for carryover. Needs a deliberate decision: reset the window at each new block, or keep
this bleed-over on purpose as a stand-in for the carryover mechanism.


In [ ]:
from collections import deque

def update_prior_reliance_smoothed(prior_reliance, window_buf, measurement_deg,
                                     prior_mean_deg, k_prior, alpha):
    """
    Same delta rule as update_prior_reliance, but agreement is judged against
    the circular mean of the last few measurements (window_buf), not the
    newest one alone -- a single noisy trial can no longer swing
    prior_reliance by itself.
    """
    window_buf.append(measurement_deg)
    smoothed_measurement = circular_mean_deg(list(window_buf))
    agreement_t = prior_agreement(smoothed_measurement, prior_mean_deg, k_prior)
    reliance_next = prior_reliance + alpha * (agreement_t - prior_reliance)
    return float(np.clip(reliance_next, 1e-4, 1.0 - 1e-4)), agreement_t


reliance = 0.5
window_buf = deque(maxlen=5)
trajectory_smoothed = []
for m, kp in zip(list(block_A) + list(block_B), [k_prior_10] * 20 + [k_prior_80] * 20):
    reliance, _ = update_prior_reliance_smoothed(reliance, window_buf, m, prior_mean_val, kp, alpha=0.2)
    trajectory_smoothed.append(reliance)

print('peak in 10-degree block:', round(max(trajectory_smoothed[:20]), 3))
print('worst dip in 10-degree block:', round(min(trajectory_smoothed[:20]), 3))


## Summary

### Decided
- Block = `(subject_id, run_id)`, stored as `block_id`
- `experiment_id` 11/12 is one combined study per subject, not two separate studies
- Model architecture: discrete mixture (reliance-weighted sum of prior and likelihood
  components), not the continuous-width multiplicative version
- `prior_reliance` is learned via a delta rule driven by per-trial agreement with the prior

### Still open
1. Real inter-block timing -- ask Salma/Varsha whether timestamps exist anywhere outside this CSV
2. Window size for smoothing `agreement_t` (tested 5, not chosen/fitted)
3. Whether the window resets at block boundaries or deliberately carries over (ties directly to
   the carryover hypothesis)
4. `alpha` -- currently a fixed demo value, needs to become a genuine fit parameter
5. Not yet built: the full per-subject trial loop, NLL, and `scipy.optimize` fitting call using
   this mechanism
6. Not yet verified: whether teammates' block-transition-labeling code (`transition_type`,
   `prev_prior_std`) already accounts for the trial-index/trial-time block-reset issue found
   above
